# 5. Logit Lens: Repression Across Network Layers

Projects hidden states through the unembedding matrix at each layer. Shows *where* in the network repression happens — and how depth predicts defence mechanism style.

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *
from pathlib import Path

theme_malign = theme_minimal() + theme(
    figure_size=(10, 6),
    plot_background=element_rect(fill='white'),
    panel_grid_minor=element_blank(),
    text=element_text(family='serif'),
    plot_title=element_text(size=14, weight='bold'),
    plot_subtitle=element_text(size=11, color='#666'),
)

# Find all logit lens CSVs for the "angry" prompt across families
data_dir = Path('data')
angry_files = sorted(data_dir.glob('logit_lens.*.she_was_so_angry*.csv'))
print(f"Found {len(angry_files)} logit lens files:")
for f in angry_files:
    print(f"  {f.name}")

In [ ]:
# Load and combine all families
frames = []
for f in angry_files:
    family = f.name.split('.')[1]
    ll = pd.read_csv(f)
    ll['family'] = family
    frames.append(ll)
df = pd.concat(frames, ignore_index=True)

# Focus on key words
key_words = ['kill', 'scream', 'punch', 'hit', 'fuck', 'kiss']
key_words = [w for w in key_words if w in df['word'].values]
dfw = df[df['word'].isin(key_words)].copy()

model_labels = {'base': 'Base (Id)', 'ego': 'SFT (Ego)', 'superego': 'DPO (Superego)'}
dfw['model_label'] = dfw['model'].map(model_labels)
dfw['model_label'] = pd.Categorical(dfw['model_label'], 
                                     categories=['Base (Id)', 'SFT (Ego)', 'DPO (Superego)'],
                                     ordered=True)

## Repression depth by family

Each panel shows one family. X-axis is network layer (0–31). Y-axis is probability (log scale). Lines show how word probabilities evolve through the network's depth.

Key contrast: OLMo suppresses `kill` from the earliest layers (distributed repression). Llama lets `kill` build up and overrides it only in the final 5 layers (late-layer redirect).

In [ ]:
# DPO model only — where does repression happen in the aligned model?
dpo = dfw[dfw.model == 'superego'].copy()
dpo['probability'] = dpo['probability'].clip(lower=1e-8)

(ggplot(dpo, aes(x='layer', y='probability', color='word'))
 + geom_line(size=1, alpha=0.8)
 + scale_y_log10()
 + scale_color_brewer(type='qual', palette='Set1')
 + facet_wrap('family', ncol=2, scales='free_y')
 + labs(title='Logit lens: where repression happens in the DPO model',
        subtitle='"She was so angry she wanted to" — probability at each network layer',
        x='Network layer', y='Probability (log scale)', color='')
 + theme_malign
 + theme(figure_size=(14, 10), strip_text=element_text(size=12, weight='bold'))
)

## Base vs DPO comparison: `kill` across layers

In [ ]:
kill = dfw[dfw.word == 'kill'].copy()
kill['probability'] = kill['probability'].clip(lower=1e-8)

(ggplot(kill, aes(x='layer', y='probability', color='model_label'))
 + geom_line(size=1.2, alpha=0.8)
 + scale_y_log10()
 + scale_color_manual(values={'Base (Id)': '#9c9c9c', 'SFT (Ego)': '#4e79a7', 'DPO (Superego)': '#e15759'})
 + facet_wrap('family', ncol=2, scales='free_y')
 + labs(title='Logit lens: "kill" across base vs aligned models',
        subtitle='Llama: kill builds up then gets overridden. OLMo: kill never builds up at all.',
        x='Network layer', y='P(kill) at each layer', color='')
 + theme_malign
 + theme(figure_size=(14, 10), strip_text=element_text(size=12, weight='bold'))
)